<a href="https://colab.research.google.com/github/OssamaSijbesma/uu-data-wrangling-assignment-1/blob/main/wc_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Assignment 1: Data Extraction & Integration](https://hqahtan.github.io/Teaching/infomdw/assignments/Assignment_1_26_27.html)

In [2]:
import sqlite3
import pandas as pd
import requests
from collections import defaultdict

# Set constants
GOOGLE_COLAB = True  # When using Google Colab set to True.
DATABASE_URL = "https://github.com/OssamaSijbesma/uu-data-wrangling-assignment-1/raw/refs/heads/main/wc_database.db"

## Attach database

In [ ]:
conn = None

if GOOGLE_COLAB:
    # Connect to in memory database
    conn = sqlite3.connect(":memory:")

    # Fetch database from online data source
    response = requests.get(DATABASE_URL)

    # Load serialized database
    conn.deserialize(response.content)
else:
    # Connect to local database file
    conn = sqlite3.connect("./wc_database.db")

## Execute queries

In [ ]:
def query_database(
    sql: str, connection: sqlite3.Connection
) -> None | list[sqlite3.Row]:
    # Create cursor
    cursor = conn.cursor()

    try:
        # Execute query
        cursor.execute(sql)

        # Fetch the output of the query
        result = cursor.fetchall()
    except sqlite3.Error as error:
        # Print error if query failed
        print(error)
        return None
    finally:
        # Close cursor to prevent memory leaks
        cursor.close()

    # Return the output as a list
    return result

In [5]:
sql_query = """SELECT pe.first_name,
    pe.last_name,
    t.country_name AS team,
    s.name AS stadium,
    g.minute
FROM goal g
    JOIN player p ON g.player_id = p.player_id
    JOIN person pe ON p.person_id = pe.person_id
    JOIN team t ON p.team_id = t.team_id
    JOIN match m ON g.match_id = m.match_id
    JOIN stadium s ON m.stadium_id = s.stadium_id
ORDER BY g.match_id,
    g.minute;"""

query_database(sql_query, conn)

[('Memphis', 'Depay', 'Netherlands', 'Johan Cruyff Arena', 23),
 ('Harry', 'Kane', 'England', 'Johan Cruyff Arena', 67),
 ('Lautaro', 'Martinez', 'Argentina', 'Camp Nou', 35),
 ('Bernardo', 'Silva', 'Spain', 'Camp Nou', 72),
 ('Harry', 'Kane', 'England', 'Wembley Stadium', 41),
 ('Lautaro', 'Martinez', 'Argentina', 'Wembley Stadium', 83)]

In [6]:
sql_query = """SELECT t.country_name,
    COUNT(g.goal_id) AS total_goals
FROM goal g
    JOIN player p ON g.player_id = p.player_id
    JOIN team t ON p.team_id = t.team_id
GROUP BY t.team_id,
    t.country_name
ORDER BY total_goals DESC;
"""

query_database(sql_query, conn)

[('England', 2), ('Argentina', 2), ('Netherlands', 1), ('Spain', 1)]

In [7]:
sql_query = "Trigger an error"

query_database(sql_query, conn)

near "Trigger": syntax error


## Read table into Pandas data frame

In [8]:
df = pd.read_sql_query("SELECT last_name, first_name, birth_date FROM PERSON", conn)
df.head()

,last_name,first_name,birth_date
0,Koeman,Ronald,1963-03-21
1,Van Dijk,Virgil,1991-07-08
2,De Jong,Frenkie,1997-05-12
3,Depay,Memphis,1994-02-13
4,Gakpo,Cody,1999-05-07


## Hash column of strings

In [ ]:
def hash_string(string: str, modulus: int) -> int:
    # Hash string to positive number
    hashed = abs(hash(string))
    # Modulo hashing
    return hashed % modulus


def hash_attributes(column: list[str], modulus: int) -> dict:
    # Create hash table
    hash_table = defaultdict(list)

    # Get the table cardinality (m)
    table_cardinality = len(column) - 1

    # Check if the modulus is between 0 and m-1
    if modulus >= table_cardinality or modulus <= 0:
        raise Exception("Modulus should be between 0 and the table cardinality - 1")

    # Loop through column
    for x in column:
        # Set the contents in the bin based on the hash
        hash_table[hash_string(x, modulus)].append(x)

    # Return the table
    return hash_table

In [57]:
# Execute hash function on data frame column
hash_table = hash_attributes(df["last_name"], 7)

# Loop to the hashed table
for key in hash_table:
    # Print key and content
    print(f"{key}: {hash_table[key]}")

4: ['Koeman', 'Kane', 'Oliver']
2: ['Van Dijk', 'Muller']
0: ['De Jong', 'Depay', 'Rice', 'Bellingham', 'Wilson']
3: ['Gakpo', 'Santos', 'Brown']
1: ['Martinez', 'Martinez', 'Taylor']
6: ['Silva', 'Garcia']


Get the index for a string

In [ ]:
print(f"The hash key for Gakpo is: {hash_string('Gakpo', 7)}")

The hash key for Gakpo is: 3


## Collision rate

In [ ]:
# initialize values
n_collisions = 0
n_elements = len(df["last_name"])

for key in hash_table:
    # Get number of elements under one index
    n_index_elements = len(hash_table[key])

    # If larger than 1 then collisions occured
    if n_index_elements > 1:
        # Get the n of elements in index and remove one
        n_collisions += n_index_elements - 1

# Calculate the collision rate
collision_rate = n_collisions / n_elements

# Print results
print(f"Total collisions: {n_collisions}")
print(f"Total elements: {n_elements}")
print(f"Collision rate: {collision_rate:.2%}")

Total collisions: 12
Total elements: 18
Collision rate: 66.67%


## Close database

In [60]:
# Close connection
conn.close()